# Model Validation: prem-1B-SQL vs T5-Large-Spider

Evaluates two open-source SQL models against our 15 scenarios using:
1. **Syntax check** — does the SQL parse without errors?
2. **Table check** — do all referenced tables exist in our schema?
3. **Column check** — do all referenced columns exist in the referenced tables?
4. **Retry simulation** — how many attempts are needed to pass all checks (max 3)?

- why am I not chcking with the direct GT SQL mathc ? - becuse we dont hvae any, need time to generate GT and avalidate it.

> No DB is needed — we validate structure only, not execution results.
> And giving all the tables in the prompt for now, no retrival.

## 1. Install Dependencies

In [1]:
# Run once
!uv add transformers torch sqlglot openpyxl pandas
# or: pip install transformers torch sqlglot openpyxl pandas

Resolved 110 packages in 9.83s                                       
⠸ Preparing packages... (0/12)                                                  ⠋ Preparing packages... (0/0)                                                   
⠸ Preparing packages... (0/12)-------------     0 B/76.54 KiB           
⠸ Preparing packages... (0/12)-------------     0 B/76.54 KiB           
typer                ------------------------------     0 B/54.77 KiB
⠸ Preparing packages... (0/12)-------------     0 B/76.54 KiB           
typer                ------------------------------     0 B/54.77 KiB
⠸ Preparing packages... (0/12)-------------     0 B/76.54 KiB           
typer                ------------------------------     0 B/54.77 KiB
⠸ Preparing packages... (0/12)-------------     0 B/76.54 KiB           
filelock             ------------------------------     0 B/25.81 KiB
typer                ------------------------------     0 B/54.77 KiB
⠸ Preparing packages... (0/12)-------------     0 B/76

## 2. Imports

In [ ]:
import time
import re
import warnings
from pathlib import Path
from collections import defaultdict

import pandas as pd
import openpyxl
import sqlglot
import sqlglot.errors

warnings.filterwarnings("ignore")

EXCEL_PATH = Path("../data/Customer Service Tables.xlsx")
MAX_RETRIES = 3

## 3. Load Schema from Excel

In [ ]:
def load_schema(excel_path: Path) -> dict[str, list[str]]:
    """Returns {table_name: [col1, col2, ...]} from the Table Metadata sheet."""
    wb = openpyxl.load_workbook(excel_path)
    ws = wb["Table Metadata"]
    schema: dict[str, list[str]] = defaultdict(list)
    for row in ws.iter_rows(min_row=2, values_only=True):  # skip header
        table_name, col_name = row[0], row[1]
        if table_name and col_name:
            schema[table_name].append(col_name.lower())
    return dict(schema)


def load_scenarios(excel_path: Path) -> list[dict]:
    """Returns list of {id, question, tables, complexity} from Scenarios sheet."""
    wb = openpyxl.load_workbook(excel_path)
    ws = wb["Scenarios"]
    scenarios = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        scenarios.append({
            "id": row[0],
            "question": row[1],
            "expected_tables": [t.strip() for t in row[2].split(",")],
            "complexity": row[3],
        })
    return scenarios


SCHEMA = load_schema(EXCEL_PATH)
SCENARIOS = load_scenarios(EXCEL_PATH)

print(f"Loaded {len(SCHEMA)} tables, {sum(len(v) for v in SCHEMA.values())} columns")
print(f"Loaded {len(SCENARIOS)} scenarios")
print("Tables:", list(SCHEMA.keys()))

## 4. SQL Validator

Three checks in order:
- **Syntax** — sqlglot parse
- **Tables** — all FROM/JOIN tables exist in our schema
- **Columns** — all selected/filtered columns exist in the referenced tables

In [ ]:
from dataclasses import dataclass, field


@dataclass
class ValidationResult:
    passed: bool
    syntax_ok: bool = False
    tables_ok: bool = False
    columns_ok: bool = False
    errors: list[str] = field(default_factory=list)


def extract_sql(raw_output: str) -> str:
    """Strip model prose and extract just the SQL statement."""
    # Try markdown code block first
    match = re.search(r"```(?:sql)?\s*([\s\S]+?)```", raw_output, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # Fallback: find first SELECT statement
    match = re.search(r"(SELECT\s+[\s\S]+?)(?:;|$)", raw_output, re.IGNORECASE)
    if match:
        return match.group(1).strip() + ";"
    return raw_output.strip()


def validate_sql(sql: str, schema: dict[str, list[str]]) -> ValidationResult:
    result = ValidationResult(passed=False)

    # --- 1. Syntax check ---
    try:
        parsed = sqlglot.parse_one(sql, dialect="sqlite")
        result.syntax_ok = True
    except sqlglot.errors.ParseError as e:
        result.errors.append(f"SYNTAX ERROR: {e}")
        return result

    # --- 2. Table check ---
    known_tables = {t.lower() for t in schema.keys()}
    referenced_tables: set[str] = set()
    table_aliases: dict[str, str] = {}  # alias -> real table name

    for node in parsed.walk():
        if isinstance(node, sqlglot.exp.Table):
            tbl_name = node.name.lower()
            alias = node.alias.lower() if node.alias else tbl_name
            if tbl_name not in known_tables:
                result.errors.append(f"UNKNOWN TABLE: '{tbl_name}'")
            else:
                referenced_tables.add(tbl_name)
                table_aliases[alias] = tbl_name

    if result.errors:
        return result
    result.tables_ok = True

    # --- 3. Column check ---
    all_referenced_cols = []
    for node in parsed.walk():
        if isinstance(node, sqlglot.exp.Column):
            col_name = node.name.lower()
            table_ref = node.table.lower() if node.table else None
            all_referenced_cols.append((col_name, table_ref))

    # Build a flat set of all valid columns across referenced tables
    all_valid_cols: set[str] = set()
    for tbl in referenced_tables:
        all_valid_cols.update(schema.get(tbl, []))

    for col_name, table_ref in all_referenced_cols:
        if col_name in ("*", ""):
            continue
        if table_ref:
            # Resolve alias to real table
            real_table = table_aliases.get(table_ref, table_ref)
            valid_cols = schema.get(real_table, [])
            if col_name not in valid_cols:
                result.errors.append(f"UNKNOWN COLUMN: '{col_name}' in table '{real_table}'")
        else:
            # No table qualifier — check across all referenced tables
            if col_name not in all_valid_cols:
                result.errors.append(f"UNKNOWN COLUMN: '{col_name}' (not in any referenced table)")

    if not result.errors:
        result.columns_ok = True
        result.passed = True

    return result


# Quick smoke test
test_sql = "SELECT t.ticket_id, c.name FROM tickets_tbl t JOIN customers_tbl c ON t.customer_id = c.customer_id WHERE t.status = 'open';"
r = validate_sql(test_sql, SCHEMA)
print(f"Smoke test: passed={r.passed}, syntax={r.syntax_ok}, tables={r.tables_ok}, columns={r.columns_ok}, errors={r.errors}")

## 5. Load Models

Both models are loaded once and reused across all scenarios.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# --- Model A: prem-1B-SQL (causal LM) ---
print("Loading prem-1B-SQL ...")
t0 = time.time()

prem_tokenizer = AutoTokenizer.from_pretrained("prem-research/prem-1B-SQL")
prem_model = AutoModelForCausalLM.from_pretrained(
    "prem-research/prem-1B-SQL",
    torch_dtype=torch.float32,
    device_map="auto",
)
prem_pipe = pipeline(
    "text-generation",
    model=prem_model,
    tokenizer=prem_tokenizer,
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=prem_tokenizer.eos_token_id,
)

print(f"prem-1B-SQL loaded in {time.time() - t0:.1f}s")

In [ ]:
# --- Model B: T5-LM-Large-text2sql-spider (seq2seq) ---
print("Loading T5-Large-Spider ...")
t0 = time.time()

t5_tokenizer = AutoTokenizer.from_pretrained("gaussalgo/T5-LM-Large-text2sql-spider")
t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "gaussalgo/T5-LM-Large-text2sql-spider",
    torch_dtype=torch.float32,
    device_map="auto",
)

print(f"T5-Large-Spider loaded in {time.time() - t0:.1f}s")

## 6. Prompt Builders

Each model expects a different input format.

In [ ]:
def schema_block(tables: list[str], schema: dict[str, list[str]]) -> str:
    """Compact schema string for prompt injection."""
    lines = []
    for tbl in tables:
        cols = ", ".join(schema.get(tbl, []))
        lines.append(f"{tbl}({cols})")
    return "\n".join(lines)


def prem_prompt(question: str, tables: list[str], schema: dict, error_hint: str = "") -> str:
    """Chat-style prompt for prem-1B-SQL (causal LM)."""
    schema_str = schema_block(tables, schema)
    error_part = f"\nPrevious attempt failed: {error_hint}\nFix and try again." if error_hint else ""
    return (
        f"### Task\nGenerate a valid SQLite SQL query for the question below.\n"
        f"Output ONLY the SQL query. No explanation, no markdown.{error_part}\n\n"
        f"### Schema\n{schema_str}\n\n"
        f"### Question\n{question}\n\n"
        f"### SQL\n"
    )


def t5_prompt(question: str, tables: list[str], schema: dict) -> str:
    """T5-Spider expects: 'translate English to SQL: {question} | {schema}'"""
    schema_str = " | ".join(
        f"{tbl} : " + " , ".join(schema.get(tbl, []))
        for tbl in tables
    )
    return f"translate English to SQL: {question} | {schema_str}"


# Sanity check
sample_tables = ["tickets_tbl", "customers_tbl"]
print("--- prem prompt ---")
print(prem_prompt("List open tickets for premium customers.", sample_tables, SCHEMA))
print("--- T5 prompt ---")
print(t5_prompt("List open tickets for premium customers.", sample_tables, SCHEMA))

## 7. Inference Wrappers

In [ ]:
def run_prem(prompt: str) -> tuple[str, float]:
    """Returns (raw_output, latency_seconds)."""
    t0 = time.time()
    outputs = prem_pipe(prompt)
    raw = outputs[0]["generated_text"]
    # Strip the input prompt from the output (causal LMs echo the input)
    raw = raw[len(prompt):].strip()
    return raw, time.time() - t0


def run_t5(prompt: str) -> tuple[str, float]:
    """Returns (raw_output, latency_seconds)."""
    t0 = time.time()
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = t5_model.generate(**inputs, max_new_tokens=256)
    raw = t5_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return raw, time.time() - t0

## 8. Retry Loop

Mirrors the real agent loop:
- Attempt 1: top-4 tables (from `expected_tables` — simulates RAG retrieval)
- Attempt 2: top-6 tables (wider retrieval fallback)
- Attempt 3: all tables in schema (full context fallback)

On each retry the validation error is fed back into the prompt.

In [ ]:
ALL_TABLES = list(SCHEMA.keys())


def get_tables_for_attempt(attempt: int, expected_tables: list[str]) -> list[str]:
    """Simulates what RAG would retrieve at each attempt."""
    if attempt == 1:
        return expected_tables  # perfect RAG (best case)
    elif attempt == 2:
        # Wider: add 2 more adjacent tables
        extra = [t for t in ALL_TABLES if t not in expected_tables][:2]
        return expected_tables + extra
    else:
        return ALL_TABLES  # full schema dump as last resort


def run_with_retries(
    scenario: dict,
    run_fn,           # run_prem or run_t5
    prompt_fn,        # prem_prompt or t5_prompt
    model_name: str,
) -> dict:
    question = scenario["question"]
    expected_tables = scenario["expected_tables"]

    attempt = 0
    total_latency = 0.0
    last_error = ""
    final_result = None

    for attempt in range(1, MAX_RETRIES + 1):
        tables = get_tables_for_attempt(attempt, expected_tables)

        if model_name == "prem":
            prompt = prompt_fn(question, tables, SCHEMA, error_hint=last_error)
        else:  # T5 doesn't use error hint (seq2seq, no chat format)
            prompt = prompt_fn(question, tables, SCHEMA)

        raw_output, latency = run_fn(prompt)
        total_latency += latency
        sql = extract_sql(raw_output)
        validation = validate_sql(sql, SCHEMA)

        if validation.passed:
            final_result = {
                "scenario_id": scenario["id"],
                "question": question,
                "complexity": scenario["complexity"],
                "model": model_name,
                "passed": True,
                "attempts": attempt,
                "sql": sql,
                "errors": [],
                "latency_s": round(total_latency, 2),
            }
            break
        else:
            last_error = "; ".join(validation.errors)

    if final_result is None:
        final_result = {
            "scenario_id": scenario["id"],
            "question": question,
            "complexity": scenario["complexity"],
            "model": model_name,
            "passed": False,
            "attempts": MAX_RETRIES,
            "sql": sql,
            "errors": validation.errors,
            "latency_s": round(total_latency, 2),
        }

    return final_result

## 9. Run All Scenarios

In [ ]:
results = []

for scenario in SCENARIOS:
    sid = scenario["id"]
    print(f"\n--- Scenario {sid} ({scenario['complexity']}) ---")
    print(f"Q: {scenario['question'][:80]}...")

    # prem-1B-SQL
    print("  [prem-1B-SQL] running...")
    prem_result = run_with_retries(scenario, run_prem, prem_prompt, model_name="prem")
    results.append(prem_result)
    status = "PASS" if prem_result["passed"] else "FAIL"
    print(f"  [prem-1B-SQL] {status} | attempts={prem_result['attempts']} | {prem_result['latency_s']}s")
    if not prem_result["passed"]:
        print(f"    errors: {prem_result['errors']}")

    # T5-Large-Spider
    print("  [T5-Large] running...")
    t5_result = run_with_retries(scenario, run_t5, t5_prompt, model_name="t5")
    results.append(t5_result)
    status = "PASS" if t5_result["passed"] else "FAIL"
    print(f"  [T5-Large]    {status} | attempts={t5_result['attempts']} | {t5_result['latency_s']}s")
    if not t5_result["passed"]:
        print(f"    errors: {t5_result['errors']}")

print("\nDone.")

## 10. Results Summary

In [ ]:
df = pd.DataFrame(results)

# --- Per-model summary ---
summary = df.groupby("model").agg(
    total_scenarios=("scenario_id", "count"),
    passed=("passed", "sum"),
    pass_rate=("passed", "mean"),
    avg_attempts=("attempts", "mean"),
    pct_first_try=("attempts", lambda x: (x == 1).mean()),
    pct_max_retries=("attempts", lambda x: (x == MAX_RETRIES).mean()),
    avg_latency_s=("latency_s", "mean"),
).round(3)

summary["pass_rate"] = summary["pass_rate"].map("{:.1%}".format)
summary["pct_first_try"] = summary["pct_first_try"].map("{:.1%}".format)
summary["pct_max_retries"] = summary["pct_max_retries"].map("{:.1%}".format)

print("=== Model Comparison ===")
print(summary.to_string())

In [ ]:
# --- Per-complexity breakdown ---
print("=== Pass Rate by Complexity ===")
complexity_summary = df.groupby(["model", "complexity"])["passed"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "passed", "count": "total", "mean": "pass_rate"}
)
complexity_summary["pass_rate"] = complexity_summary["pass_rate"].map("{:.1%}".format)
print(complexity_summary.to_string())

In [ ]:
# --- Error type breakdown ---
print("=== Validation Error Types ===")
for model in ["prem", "t5"]:
    failed = df[(df["model"] == model) & (~df["passed"])]
    syntax_errs = sum(1 for errors in failed["errors"] for e in errors if "SYNTAX" in e)
    table_errs = sum(1 for errors in failed["errors"] for e in errors if "TABLE" in e)
    col_errs = sum(1 for errors in failed["errors"] for e in errors if "COLUMN" in e)
    print(f"\n{model}:")
    print(f"  Failed scenarios : {len(failed)}")
    print(f"  Syntax errors    : {syntax_errs}")
    print(f"  Unknown tables   : {table_errs}")
    print(f"  Unknown columns  : {col_errs}")

In [ ]:
# --- Scenario-level detail table ---
print("=== Per-Scenario Results ===")
pivot = df.pivot(index="scenario_id", columns="model", values=["passed", "attempts", "latency_s"])
pivot.columns = [f"{col[1]}_{col[0]}" for col in pivot.columns]
pivot["question"] = [s["question"][:60] + "..." for s in SCENARIOS]
pivot["complexity"] = [s["complexity"] for s in SCENARIOS]
print(pivot[["question", "complexity", "prem_passed", "prem_attempts", "prem_latency_s", "t5_passed", "t5_attempts", "t5_latency_s"]].to_string())

## 11. Decision: Which Model to Use?

Based on the results above, fill in this decision matrix:

| Criterion | prem-1B-SQL | T5-Large-Spider |
|---|---|---|
| Pass rate (all scenarios) | _fill after run_ | _fill after run_ |
| Pass rate (Hard only) | _fill after run_ | _fill after run_ |
| % passed on first attempt | _fill after run_ | _fill after run_ |
| % hit max retries (bad signal) | _fill after run_ | _fill after run_ |
| Avg latency per request | _fill after run_ | _fill after run_ |
| Multi-JOIN support | Yes | Limited |
| Error hint support (retry prompt) | Yes | No (seq2seq) |
| Model size | 1.3B | 770M |

**Decision threshold:**
- If `pct_max_retries > 20%` → upgrade to XiYanSQL-3B or Claude API.
- If T5 pass rate < prem pass rate by >10% → drop T5, use prem only.

In [ ]:
# Auto-print the decision recommendation
prem_stats = summary.loc["prem"]
t5_stats = summary.loc["t5"]

prem_retry_rate = df[(df["model"] == "prem") & (df["attempts"] == MAX_RETRIES)].shape[0] / 15
t5_retry_rate = df[(df["model"] == "t5") & (df["attempts"] == MAX_RETRIES)].shape[0] / 15

print("=== Recommendation ===")

if prem_retry_rate > 0.20:
    print("WARNING: prem-1B-SQL max-retry rate > 20%. Consider upgrading to XiYanSQL-3B.")
else:
    print("OK: prem-1B-SQL max-retry rate is within threshold.")

if t5_retry_rate > 0.20:
    print("WARNING: T5-Large max-retry rate > 20%. T5 may not be suitable for complex queries.")

prem_pass = df[df["model"] == "prem"]["passed"].mean()
t5_pass = df[df["model"] == "t5"]["passed"].mean()

if prem_pass - t5_pass > 0.10:
    print(f"\nPrem-1B outperforms T5 by {(prem_pass - t5_pass):.0%} pass rate → USE prem-1B-SQL.")
elif t5_pass - prem_pass > 0.10:
    print(f"\nT5 outperforms prem-1B by {(t5_pass - prem_pass):.0%} pass rate → USE T5-Large-Spider.")
else:
    print(f"\nModels are close ({prem_pass:.0%} vs {t5_pass:.0%}). Use prem-1B-SQL for retry hint support.")